# 一个大模型怎么拆到几百张卡上？5D 并行

一个 100B 的模型，每个参数用 2 字节存，参数个数 × 字节数 = 1000 亿 × 2 = 200GB。而一张 H100 只有 80GB 显存。

一张卡装不下，就只能拆。问题是往哪拆。模型的结构决定了能拆的地方有限：数据可以切、权重可以切、层可以切、序列可以切、专家可以切，正好五种。

这一节我们逐个看这五种拆法：切什么、为什么、有什么代价。


## 1. 数据并行 DP：最简单，但显存不省

**切的对象**：训练数据。

假设你有 8 张卡，一个训练 batch 是 64 个样本。DP 做的事很简单：把 64 个样本平均切成 8 份，每张卡拿到 8 个样本，然后**每张卡各自持有一份完整的模型**，同时算自己的那 8 个样本。

**训练流程**（DDP 的四个步骤）：

1. 每张卡拿自己那份数据做前向、反向，算出**自己的梯度**；
2. all-reduce 把所有卡的梯度求平均（附录 E 讲过，人人拿到相同梯度）；
3. 每张卡用相同的平均梯度更新自己的参数；
4. 由于每张卡初始参数一样、梯度一样、更新规则一样，参数始终一致。

DP 换来的是**吞吐量 ×N**：8 张卡同时处理 8 倍的数据。
但它的软肋很明显：**显存完全没省**——模型有多大，每张卡就得放多大。

模型小的时候 DP 是最优解；模型大到一张卡放不下，就要想办法把模型也切开。
切开模型之前，先看看 ZeRO：它把 DP 里的冗余（优化器状态、梯度、参数）逐步切开，
单卡显存大幅下降，而通信量几乎不变。



In [ ]:
# === ZeRO 三个 Stage 的单卡显存：7B 模型 × 8 卡 ===
# 一个 BF16 模型，训练时每参数大约占 16 字节：
#   参数 2 + 梯度 2 + 优化器状态 12（Adam 的 m 和 v 各 4 字节 + 主权重 4）
P = 7e9        # 参数个数
N = 8          # 卡数

ddp_per_card   = 16 * P                        # 每卡完整副本
zero1_per_card = 2 * P + 2 * P + 12 * P / N    # 只切优化器状态
zero2_per_card = 2 * P + 2 * P / N + 12 * P / N  # 切优化器 + 梯度
zero3_per_card = 16 * P / N                    # 全切

print(f"{'方案':<14}{'单卡 (GB)':>12}{'相对 DDP':>12}")
print("-" * 40)
for name, val in [("DDP", ddp_per_card),
                  ("ZeRO-1", zero1_per_card),
                  ("ZeRO-2", zero2_per_card),
                  ("ZeRO-3", zero3_per_card)]:
    print(f"{name:<14}{val/1e9:>10.1f}   {val/ddp_per_card*100:>8.1f}%")

print()
print("关键观察：ZeRO-3 把单卡显存从 112 GB 压到 14 GB（1/8）。")
print("但代价是训练时要反复 all-gather 参数、reduce-scatter 梯度，对带宽更敏感。")

### 1.1 记住三个数字

上面代码里用了「16 字节/参数」这个数，它从哪来？训练一个 BF16 模型，每个参数在训练时大约占 **16 字节**：

- 参数本身 2 字节（BF16）；
- 梯度 2 字节；
- 优化器状态 12 字节（Adam 需要保存动量 m、二阶矩 v，还有一份高精度主权重）。

为什么要这么多？因为反向传播要先算梯度，梯度本身就是一份数据；Adam 优化器还要额外记住每个参数过去的动量 m 和二阶矩 v，并且为了更新更稳，还要保留一份高精度的主权重。这几样加起来，比参数本身多得多。

这个「16 字节/参数」以后随手就能用来估算显存：
7B 模型 × 16 字节 ≈ 112 GB，和上面表格第一行对上了。



## 2. Tensor Parallelism：把一层切开（TP）

DP 复制模型不省显存，ZeRO 省了显存但每步要频繁通信。还有没有别的思路？

有——**把每一层的权重切成 N 片，N 张卡各算一片**。这就是 Tensor Parallelism（TP）。

**关键问题**：矩阵乘法怎么切？一个线性层 $y = xW$，权重 W 是 $[d_{in}, d_{out}]$ 的矩阵，
可以沿两个方向切：

- **Column 切法**：把 W 按**输出维度**（列）切成 N 片，每张卡持有 $[d_{in}, d_{out}/N]$。
  每张卡算出输出的 1/N 列，**不需要通信**（大家输入是同一份 x）。
- **Row 切法**：把 W 按**输入维度**（行）切成 N 片，每张卡持有 $[d_{in}/N, d_{out}]$。
  每张卡算出的是**部分和**（因为每一列的输出 = 各行输入 × 权重的和），**需要 all-reduce** 才能得到完整结果。

为什么两种切法一个免通信、一个要通信？想想矩阵乘法的定义。Column 切法下，每张卡拿的是同一份输入 x、不同的几列权重，它算出的就是最终输出里对应的那几列——各自独立，谁也不用等谁。而 Row 切法下，每张卡拿的是 x 里不同的几行、全部的列权重，它算出的只是「对最终输出的部分贡献」；要把每一列凑齐，必须把所有卡的部分和加在一起，这一步就是 all-reduce。

看起来 Column 不通信、Row 要通信。但注意：Column 切法的输出是「部分列」，
如果下一层要的是完整输出，还是得拼回来（相当于还是要一次通信）。

所以工业界的做法是把两者**组合**起来，下面我们手算验证这个组合。



In [ ]:
# === 先看两种切法长什么样（2 卡，小矩阵） ===
import numpy as np
np.random.seed(0)

# 一个线性层 y = xW，输入 4 维，输出 6 维
d_in, d_out = 4, 6
W = np.random.randn(d_in, d_out)   # 权重 [4, 6]

# Column 切法：按输出维度（列）切成 2 片，每片 [4, 3]
col_shards = np.split(W, 2, axis=1)
print("Column 切法（按输出维度切，每卡拿几列）：")
for r, s in enumerate(col_shards):
    print(f"  rank {r} 权重形状: {s.shape}")

# Row 切法：按输入维度（行）切成 2 片，每片 [2, 6]
row_shards = np.split(W, 2, axis=0)
print("\nRow 切法（按输入维度切，每卡拿几行）：")
for r, s in enumerate(row_shards):
    print(f"  rank {r} 权重形状: {s.shape}")

print()
print("关键观察：不管哪种切法，参数总量不变，只是换了个方向切开。")
print("下一步看这两种切法怎么组合成完整的 MLP。")

### 2.1 关键组合：column → 激活 → row，只需一次 all-reduce

上面已经知道：Column 切法免通信，Row 切法要 all-reduce。那到底怎么用，才能把一次 MLP 的通信压到最少？

一个 MLP 有两个线性层：第一层 $W_1$（把维度从小变大），第二层 $W_2$（把维度从大变回小），
中间夹一个激活函数。Megatron-LM 的精妙之处在于把两层**错开切**：

- **第一层 $W_1$ 用 Column 切法**（按输出切），每张卡算出自己那批中间列，无需通信；
- 激活函数（如 SiLU）是**逐元素**的，可以安全地在每张卡上各自算；
- **第二层 $W_2$ 用 Row 切法**（按输入切），每张卡算出部分和，**最后 all-reduce 一次**。

于是整层 MLP 从输入到输出，**只需要一次 all-reduce**。

为什么对？关键在第二层。第一层按列切，正好让每张卡手里握着「中间向量中的不同几列」；第二层按行切，正好让每张卡把这些列算成对最终输出的部分和。两层一个按列、一个按行，正好「咬合」起来，中间的激活函数又是逐元素的，所以在中间不需要任何通信，全部通信被省到了最后那一次 all-reduce 上。

下面用具体数字验证一次。



In [ ]:
# === 手算验证：2 卡 TP 的 MLP 只需一次 all-reduce ===
import numpy as np
np.random.seed(0)

d_model = 4     # 输入维度
hidden  = 6     # 中间维度
world_size = 2

W1 = np.random.randn(d_model, hidden)   # 第一层 [4, 6]
W2 = np.random.randn(hidden, d_model)   # 第二层 [6, 4]
x  = np.random.randn(1, d_model)        # 输入 [1, 4]


def silu(z):
    """SiLU 激活函数（逐元素，可以直接在每张卡上算）。"""
    return z / (1.0 + np.exp(-z))

# 参考输出：单卡上完整计算
y_ref = silu(x @ W1) @ W2

# TP 切法：W1 按列切（Column），W2 按行切（Row）
W1_shards = np.split(W1, world_size, axis=1)   # 每片 [4, 3]
W2_shards = np.split(W2, world_size, axis=0)   # 每片 [3, 4]

partials = []
for r in range(world_size):
    h_local = silu(x @ W1_shards[r])   # 每卡算自己的中间列，无需通信
    y_partial = h_local @ W2_shards[r] # 每卡算部分和
    partials.append(y_partial)
    print(f"rank {r}: 局部输出 shape {y_partial.shape}, 数值 {y_partial.ravel()}")

# 模拟一次 all-reduce（求和）
y_tp = sum(partials)
print()
print(f"参考输出 y_ref: {y_ref.ravel()}")
print(f"TP 重建   y_tp: {y_tp.ravel()}")
print(f"最大误差: {np.abs(y_ref - y_tp).max():.2e}")

print()
print("关键观察：column → 激活 → row 整条路径，只在结尾做了一次 all-reduce。")
print("这就是 TP 的核心：通信次数固定，不随层数增长而翻倍。")

### 2.2 Attention 怎么切：按 head 切，天然免通信

Transformer 的另一半是 attention。它怎么切？总不能让多张卡一起算一个 attention 吧？

答案是**按 head 切**。attention 里每个 head 是独立的：head A 的 Q 只和自己 head 的 K、V 做点积，
不和其他 head 有任何瓜葛。这是一个天然的切分边界——head 与 head 之间没有任何交叉计算。所以把 32 个 head 平均分到 4 张卡，每张卡独立算自己的 8 个 head，
**attention 内部完全不需要通信**。

切法和 MLP 一样：QKV 投影用 Column 切法（按输出切，把 3×d_model 的输出切成 N 片），
输出投影用 Row 切法，最后一次 all-reduce。

工程要求：head 数要能被卡数整除（比如 32 head ÷ 4 卡 = 每卡 8 head）。



In [ ]:
# === attention 按 head 切：验证每卡独立算、最后求和 ===
import numpy as np
np.random.seed(2)

d_model, num_heads = 8, 4
head_dim = d_model // num_heads    # 每 head 2 维
seq_len = 3
world_size = 2                     # 2 卡，每卡 2 个 head

x = np.random.randn(1, seq_len, d_model)
# 每个 head 独立的权重（这是 MHA 的等价写法，方便按 head 切分）
W_qkv = np.random.randn(num_heads, d_model, 3 * head_dim)
W_out = np.random.randn(num_heads, head_dim, d_model)


def softmax(z):
    e = np.exp(z - z.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)


def head_forward(x, h):
    """单个 head 的完整前向：投影 → 点积 → softmax → 输出投影。"""
    qkv = (x @ W_qkv[h]).reshape(1, seq_len, 3, head_dim)
    q, k, v = qkv[..., 0, :], qkv[..., 1, :], qkv[..., 2, :]
    sc = q @ k.transpose(0, 2, 1) / np.sqrt(head_dim)
    att = softmax(sc) @ v
    return att @ W_out[h]

# 参考：所有 head 加起来（等价于完整 MHA）
y_ref = sum(head_forward(x, h) for h in range(num_heads))

# TP 切法：把 head 分到 2 张卡，每卡 2 个 head，零通信
heads_per_rank = num_heads // world_size
partials = []
for r in range(world_size):
    local = range(r * heads_per_rank, (r + 1) * heads_per_rank)
    partial = sum(head_forward(x, h) for h in local)
    partials.append(partial)

y_tp = sum(partials)   # 结尾一次 all-reduce

print(f"参考 y_ref[0,0,:] = {y_ref[0,0,:]}")
print(f"TP 重建 y_tp[0,0,:] = {y_tp[0,0,:]}")
print(f"最大误差 = {np.abs(y_ref - y_tp).max():.2e}")

print()
print("关键观察：每张卡独立算自己那批 head，attention 内部零通信。")
print("TP 在 attention 上的优雅之处：切在 head 上，正好切在自然的边界上。")

### 2.3 TP 的代价：通信频繁，只能留在节点内

TP 的优点是显存随卡数线性下降，但代价也明确：**每个 transformer 层都要通信 4 次**。

- MLP：前向结尾一次 all-reduce + 反向一次；
- Attention：前向结尾一次 all-reduce + 反向一次。

为什么反向也要通信？因为反向传播要算梯度，而梯度的形状和 forward 的 all-reduce 是对称的——forward 里怎么把部分和拼起来，backward 里就得怎么把梯度拆回去。所以每一个 forward 的 all-reduce，都对应一个 backward 的 all-reduce。MLP 和 Attention 各一对，加起来就是 4 次。

每层 4 次，而且都在计算的关键路径上——算完就要等通信。
所以 TP 的通信必须走最快的 NVLink（450 GB/s），**几乎只能在一个节点内用**（8 卡）。
跨节点做 TP 通信会慢到无法接受，跨节点的活儿交给流水并行 PP 或数据并行 DP。

下面看不同 TP 大小下单卡权重和通信次数的对比。



In [ ]:
# === TP 越大，单卡权重越少，但通信次数不变 ===
d_model, d_ff = 4096, 11008
bytes_elem = 2   # BF16

print(f"{'TP':>4}{'单卡 MLP 权重 (MB)':>20}{'all-reduce 次数/层':>20}")
print("-" * 46)
for tp in [1, 2, 4, 8]:
    w_bytes = (d_model * d_ff + d_ff * d_model) * bytes_elem / tp
    comms = 4 if tp > 1 else 0   # fwd+bwd × (MLP+Attn)，TP=1 时无通信
    print(f"{tp:>4}{w_bytes / 1e6:>18.1f}{comms:>20}")

print()
print("关键观察：TP=8 时单卡权重压到 1/8，但每层仍是 4 次 all-reduce。")
print("通信次数不随 TP 增大而减少——所以 TP 只适合节点内（NVLink 快）。")

## 3. Pipeline Parallelism：把层切开（PP）

TP 是把**一层**切开，还有一种更直觉的切法：把**层**切开——第 1 到第 5 层放卡 0，
第 6 到第 10 层放卡 1……这就是 Pipeline Parallelism（PP）。

**最朴素的想法**：一层一张卡，前向从卡 0 传到卡 1 再传到卡 2……
这有什么问题？一批数据要依次穿过卡 0、卡 1、卡 2……，**任何时刻只有一张卡在算，其他卡都在等**。
把 4 层分到 4 张卡，速度和 1 张卡一样，只是显存分摊了——这不划算。

### 3.1 GPipe 的改进：把一批拆成小批，错峰执行

问题出在「一批数据一次只经过一张卡」。那如果一次只传一个小批，让它们像流水线一样错开，是不是就能让所有卡同时忙起来？

GPipe 的想法：把一个 batch 切成 M 个 **micro-batch**（小批），让它们在流水线上**错峰**执行。
卡 0 算完 micro-batch 0 立刻开始算 micro-batch 1，同时卡 1 在算 micro-batch 0……
这样大部分时间所有卡都在干活，只有流水线开头和结尾有人闲着（这闲着的部分叫 **bubble**，空泡）。

空泡占比 = $\frac{P-1}{M}$（P 是层数/卡数，M 是 micro-batch 数）。
M 越大，空泡越小。



In [ ]:
# === 手算：bubble（空泡）占比 = (P-1)/M ===
print(f"{'P (层数)':>10}{'M (小批数)':>12}{'bubble 占比':>14}")
print("-" * 38)
for P in [4, 8]:
    for M in [8, 32, 128]:
        bubble = (P - 1) / M
        print(f"{P:>10}{M:>12}{bubble*100:>12.1f}%")

print()
print("关键观察：P=8 时，M=128 空泡只有 5.5%，几乎可以忽略。")
print("所以工程上 M 通常取层数的 4~8 倍以上。")
print("注意：M 必须大于 P-1，否则公式给出荒谬的大数——小批太少，流水线根本排不满。")

### 3.2 直观感受：错峰的流水线时间线

嘴上说「错峰」有点抽象，下面用字符画直接看。P=4 层、M=4 个小批，
每个格子是一个时间单位，F0 表示 micro-batch 0 的前向，B0 表示它的反向，`.` 表示空闲。
（简化示意：真实调度还有 1F1B 等变体，这里先看 GPipe 的交错思想。）



In [ ]:
# === 流水线时间线字符画（P=4, M=4，简化示意） ===
P, M = 4, 4

def line(stage):
    """生成一个 stage 的时间线字符串：先等前面塞满，再 F，最后 B。"""
    cells = [" . "] * stage          # 等前面的层先开始
    cells += [f"F{i}" for i in range(M)]   # 处理 M 个小批的前向
    cells += [" . "] * (P - 1 - stage)     # 结尾的空泡（示意）
    cells += [f"B{i}" for i in range(M)]   # 反向
    return cells

print("GPipe 交错流水线（P=4, M=4；F=前向，B=反向，. = 空闲）")
print("=" * 62)
for s in range(P):
    print(f"stage {s}: " + " ".join(line(s)))

print()
print("关键观察：越靠后的层，开头等得越久（warmup 空泡）；")
print("所有层的结尾都有一段空闲（cooldown 空泡）。")
print("中间大部分时间所有人都在干活——这就是流水线的价值。")

### 3.3 进阶：1F1B 和 DualPipe（知道名字即可）

GPipe 的做法是「一个 stage 先把所有前向做完，再做所有反向」，这样要保存很多激活值。
为什么？因为反向传播要用到前向时算出来的中间结果，如果前向和反向隔得很远，就得把中间结果都存下来，等反向来取——存的越多，显存越紧张。

工业界的改进有两个：

- **1F1B**（one forward, one backward）：每个 stage 交替做前向和反向，
  一个前向接一个反向，激活值内存下降，流水线依然排满；
- **DualPipe**（DeepSeek-V3 提出）：把 micro-batch 分成两半，一半从开头往结尾走，
  一半从结尾往开头走，双向交错，进一步压掉两端的空泡。代价是显存占用更高。

这两个的名字后面读技术报告时还会见到。这一节只需要知道：**PP 的核心矛盾是
「空泡」和「激活显存」之间的取舍，工业界一直在优化。**



## 4. Sequence Parallelism：把序列切开（SP）

前面切数据、切权重、切层，都是「模型放不下」时的办法。还有一类需求：
**模型放得下，但输入太长**——比如 128K 的上下文。

为什么输入长会出问题？attention 的中间激活和序列长度呈平方关系（QKᵀ 是一张「序列长 × 序列长」的矩阵），序列一长，激活值就爆炸式增长，最先塞满的往往不是权重，而是这些中间激活。这时单张卡存不下，就要把**序列长度**切开，这就是 Sequence Parallelism（SP）。

**两个用途**：

1. **配合 TP 省显存**：LayerNorm、dropout 这类操作对每个 token 独立，
   没必要每张卡都持完整的序列——把序列也切了，每卡只算自己那批 token；
2. **真正支持超长上下文**：序列长到一张卡放不下激活时，把序列切到多卡，
   用 Ring Attention 之类的方法让每张卡算自己那段与其它段的关系。

下面算一下切序列对显存的帮助有多大。



In [ ]:
# === SP：切序列对 LayerNorm 激活显存的影响 ===
batch, seq_len, d_model = 1, 8192, 4096
bytes_elem = 2

full_bytes = batch * seq_len * d_model * bytes_elem
print(f"完整 LayerNorm 输入激活: {full_bytes / 1e6:.1f} MB")
print(f"{'SP':>6}{'单卡激活 (MB)':>18}{'节省':>10}")
print("-" * 36)
for sp in [1, 2, 4, 8]:
    per_card = full_bytes / sp
    print(f"{sp:>6}{per_card / 1e6:>16.1f}{(1 - 1/sp)*100:>8.0f}%")

print()
print("关键观察：SP=8 时单卡激活压到 1/8。")
print("代价：进入 TP 前要 all-gather 拼回序列，退出后再 reduce-scatter 切回去。")

## 5. Expert Parallelism：把专家切开（EP）

第五种并行只针对一类特殊模型：**MoE**（Mixture of Experts，混合专家）。

先花 30 秒认识 MoE：它把 transformer 里的 FFN 层替换成**一组并行的专家网络**。
输入经过一个 router（路由器），每个 token 只被送到 top-k 个最相关的专家去算。
这样总参数可以很大（几百个专家），但每个 token 只激活一小部分，计算量不大。

**问题**：专家太多，一张卡放不下怎么办？注意这里的「放不下」指的是权重——MoE 的总参数分散在几百个专家里，虽然每个 token 只用其中几个，但所有专家都得存着。专家一多，一张卡的显存就装不下了。

**EP 的做法**：把专家**切分到各卡**——卡 0 放专家 0~7，卡 1 放专家 8~15……
每张卡只存自己那批专家。这里「切」的对象是专家本身，而不是把单个专家再切细。

**关键通信**：token 要去「它选中的专家」所在的卡计算。
这就用到了附录 E 讲的 **all-to-all**：token 从自己的卡，被发到专家所在的卡；
算完之后，结果再 all-to-all 发回来（这叫 combine）。

下面模拟一次 dispatch（发送 token 给专家）。



In [ ]:
# === EP 模拟：4 个专家分到 4 张卡，看 token 怎么 dispatch ===
import numpy as np

num_experts = 4
world_size = 4
seq_len = 8

d_model = 4
np.random.seed(1)
tokens = np.random.randn(seq_len, d_model)          # 8 个 token
router_logits = tokens @ np.random.randn(d_model, num_experts)
assignments = router_logits.argmax(axis=1)          # 每个 token 选一个专家

print(f"token → expert 路由: {assignments.tolist()}")

# 模拟 all-to-all：每个专家（卡）收集发给自己的 token
expert_inputs = [[] for _ in range(num_experts)]
for tok_idx, expert_id in enumerate(assignments):
    expert_inputs[expert_id].append(tokens[tok_idx])

for eid in range(num_experts):
    print(f"expert {eid}（卡 {eid}）收到 {len(expert_inputs[eid])} 个 token")

print()
print("关键观察：token 从自己的卡跑到专家所在的卡——这就是 all-to-all。")
print("算完之后，结果还要按原路 all-to-all 发回去（combine）。")

In [ ]:
# === EP vs TP vs DP 在 MoE 上的单卡显存对比 ===
# 假设 8 个 expert × 每个 expert 110M 参数，共 8 张卡
expert_params = 110e6
num_experts = 8
total_params = expert_params * num_experts
N = 8

ep_per_card = expert_params        # EP：每卡 1 个 expert
pp_same = total_params / N          # 若把每个 expert 都切成 8 片（每卡每 expert 1/8）
dp_per_card = total_params          # DP：每卡完整副本

print(f"{'方案':<10}{'单卡 expert 权重 (M)':>22}{'通信模式':>16}")
print("-" * 50)
print(f"{'EP':<10}{ep_per_card/1e6:>20.1f}{'all-to-all':>16}")
print(f"{'TP':<10}{pp_same/1e6:>20.1f}{'all-reduce':>16}")
print(f"{'DP':<10}{dp_per_card/1e6:>20.1f}{'all-reduce (梯度)':>16}")

print()
print("关键观察：EP 单卡最省，但用 all-to-all（无法用 ring 优化），")
print("所以 EP 对网络拓扑要求最高——这也是上一节硬件那节强调的。")

### 5.1 EP 的通信量有多大

EP 的通信量可以精确估算。每个 token 要发到它激活的 expert 所在的卡，
一次 dispatch + 一次 combine，每个 MoE 层 2 次 all-to-all。

以 Mixtral 8×7B 为例算一笔（只是数量级感受）：



In [ ]:
# === Mixtral 8x7B 一个 MoE 层的 all-to-all 通信量 ===
batch = 1024       # 全局 batch
seq = 4096
hidden = 4096
top_k = 2
n_ep = 8           # EP 并行度
bytes_elem = 2     # BF16

# 每卡发出的 token 数 = (batch×seq) / n_ep × top_k（每个 token 激活 2 个专家）
tokens_per_card = batch * seq / n_ep * top_k
bytes_per_send = tokens_per_card * hidden * bytes_elem

print(f"Mixtral 8x7B（EP={n_ep}, top_k={top_k}）：")
print(f"  每卡一次 all-to-all 发送: {bytes_per_send / 1e9:.2f} GB")
print(f"  一个 MoE 层（dispatch+combine 共 2 次）: {2 * bytes_per_send / 1e9:.2f} GB")
print(f"  32 层 MoE 总计: {32 * 2 * bytes_per_send / 1e9:.1f} GB")
print()
print("关键观察：一个训练 step 光 MoE 通信就要几百 GB，")
print("这还是在 EP=8 的高并行度下。EP 越大，单卡发送量越小，但延迟越高。")

## 6. 五种并行怎么组合：真实模型怎么配

五种并行都学完了。真实的大模型训练，**不是选一种，而是同时用好几种**，
组成一个并行配置（recipe）。

为什么能同时用？因为五种并行切的是不同的东西——数据、权重、层、序列、专家——互不冲突，可以层层嵌套。一个 2048 卡集群的典型组织方式：

- **最内层 8 卡**：TP=8（必须在一个节点内，走 NVLink）；
- **中间层**：EP 或 PP（跨节点，但尽量在相邻节点）；
- **最外层**：DP（复制整组，扩展吞吐量）。

通信最快的放最内层，通信慢的放外层，这样每一层都只在自己的通信半径内干活。下面看看三个公开报告的配置：



In [ ]:
# === 三个真实模型的并行配置 ===
# 总 GPU 数 = TP × PP × DP × EP × SP
configs = [
    ("Llama-3 70B",  "dense", {"TP": 8,  "PP": 8,  "DP": 16, "EP": 1,  "SP": 1}),
    ("Mixtral 8x7B", "MoE",   {"TP": 1,  "PP": 1,  "DP": 16, "EP": 8,  "SP": 1}),
    ("DeepSeek-V3",  "MoE",   {"TP": 1,  "PP": 16, "DP": 2,  "EP": 64, "SP": 1}),
]

print(f"{'模型':<16}{'类型':<8}{'TP':>4}{'PP':>5}{'DP':>5}{'EP':>5}{'SP':>5}{'总 GPU':>10}")
print("-" * 60)
for name, kind, c in configs:
    total = c["TP"] * c["PP"] * c["DP"] * c["EP"] * c["SP"]
    print(f"{name:<16}{kind:<8}{c['TP']:>4}{c['PP']:>5}{c['DP']:>5}{c['EP']:>5}{c['SP']:>5}{total:>10}")

print()
print("怎么读这三行：")
print("  Llama-3 是 dense 模型，没专家，靠 TP(节点内) + PP + DP。")
print("  Mixtral 是 MoE，EP=8 让 8 个专家各占一张卡，其余靠 DP。")
print("  DeepSeek-V3 有 256 个专家，EP=64 是主力；PP=16 用 DualPipe 调度。")

### 6.1 组合的三条原则

看了三个例子，背后其实是三条通用原则：

1. **TP 只在节点内用**（通信频繁，必须走 NVLink）；
2. **EP 只用于 MoE**，且专家数要和 EP 并行度匹配；
3. **PP 和 DP 填剩下的 GPU**，且 micro-batch 数要足够大以掩盖空泡。

模型越大、专家越多，EP 的占比就越高。这也是为什么大 MoE 模型（DeepSeek 系列）
特别吃网络带宽——EP 的 all-to-all 对拓扑极其敏感（回到第一节硬件那节）。



## 小结

这一节的逻辑链：

- 模型放不下 → 需要「分」→ 五种分法（5D 并行）；
- DP 最简单但显存不省，ZeRO 把 DP 里的冗余切开；
- TP 切一层内的权重，column→row 组合后每层只需一次 all-reduce，但只能节点内用；
- PP 切层，靠 micro-batch 错峰掩盖空泡，空泡 = (P-1)/M；
- SP 切序列，解决长上下文；EP 切专家，靠 all-to-all，MoE 专用；
- 真实训练是多种并行组合，总 GPU 数 = TP×PP×DP×SP×EP。

确认你已经搞懂：

- [ ] DP 切数据，每卡有完整模型副本，吞吐 ×N 但显存不省
- [ ] ZeRO 三个 Stage 通信量相同，单卡显存从 16P 降到 16P/N
- [ ] TP 的 Column 切法按输出切（无需通信），Row 切法按输入切（要 all-reduce）
- [ ] column → 激活 → row 组合，整层 MLP 只需一次 all-reduce
- [ ] attention 按 head 切，head 之间天然独立，内部零通信
- [ ] TP 每层 4 次 all-reduce，只能放在节点内（NVLink）
- [ ] PP 切层，bubble 占比 = (P-1)/M，M 越大空泡越小
- [ ] SP 切序列，主要用于长上下文训练
- [ ] EP 切专家，核心通信是 all-to-all（dispatch + combine）
- [ ] 总 GPU 数 = TP × PP × DP × SP × EP，真实模型是多种并行的组合



## 作业

> 可以让 AI 帮忙解释思路、检查方向，但不建议直接让 AI「做完这道题」。

**作业 1：TP=2 + DP=2 下单卡显存**

7B dense 模型，BF16 + AdamW 训练，4 张卡。配置 TP=2、DP=2，不开 ZeRO。
每张卡持有什么？单卡固定显存是多少 GB？

小提示：TP=2 把权重和梯度切成一半（2P/2 每项）；优化器状态没有 ZeRO 帮忙，
仍然是 12P。总 = 2P/2 + 2P/2 + 12P = 14P 字节。



In [ ]:
# 作业 1：TP=2 + DP=2 下 7B 模型的单卡显存
P = 7e9

# TODO: 单卡显存（字节）
per_card_bytes = None

assert per_card_bytes is not None, "请先计算单卡显存"
expected = 2 * P / 2 + 2 * P / 2 + 12 * P   # = 14P
assert abs(per_card_bytes - expected) < 1e9, f"应为 {expected / 1e9:.0f} GB"

print(f"作业 1 通过！")
print(f"  TP=2 + DP=2 下单卡显存 = {per_card_bytes / 1e9:.0f} GB")
print(f"  对比纯 DDP 的 112 GB：权重和梯度被 TP 切了一半，优化器状态没省。")

**作业 2：算 PP 的空泡占比**

PP=8 层、M=32 个小批，空泡占比是多少？（用公式 (P-1)/M 算百分比）

小提示：直接代公式。顺便想想：M 增大到 128，空泡变成多少？



In [ ]:
# 作业 2：PP bubble 占比
P, M = 8, 32

# TODO: 计算空泡占比（0~1 之间的小数）
bubble_ratio = None

assert bubble_ratio is not None, "请先计算空泡占比"
expected = (P - 1) / M
assert abs(bubble_ratio - expected) < 1e-6, f"应为 {expected:.4f}"

print(f"作业 2 通过！")
print(f"  PP={P}、M={M} 时空泡占比 = {bubble_ratio*100:.1f}%")
print(f"  如果把 M 提高到 128，空泡会降到 {(P-1)/128*100:.1f}%。")
print(f"  工程上常用「micro-batch 数 = 层数 × 4~8」来压低空泡。")

**作业 3：DeepSeek-V3 的总 GPU 数**

DeepSeek-V3 的配置是 TP=1、PP=16、DP=2、EP=64、SP=1。总 GPU 数是多少？

小提示：总 GPU = TP × PP × DP × EP × SP，直接相乘。



In [ ]:
# 作业 3：DeepSeek-V3 总 GPU 数
config = {"TP": 1, "PP": 16, "DP": 2, "EP": 64, "SP": 1}

# TODO: 计算总 GPU 数
total_gpus = None

assert total_gpus is not None, "请先计算总 GPU 数"
expected = config["TP"] * config["PP"] * config["DP"] * config["EP"] * config["SP"]
assert total_gpus == expected, f"应为 {expected}"

print(f"作业 3 通过！")
print(f"  DeepSeek-V3 配置: {config}")
print(f"  总 GPU = {total_gpus}（公开报告是 2048 张 H800）")
print(f"  EP=64 是主力——256 个专家分到 64 张卡，每卡 4 个专家。")

## 参考资料

- Shoeybi et al., [Megatron-LM: Training Multi-Billion Parameter Language Models Using Model Parallelism](https://arxiv.org/abs/1909.08053), 2019
- Huang et al., [GPipe: Efficient Training of Giant Neural Networks using Pipeline Parallelism](https://arxiv.org/abs/1811.06965), 2018
- Narayanan et al., [Memory-Efficient Pipeline-Parallel DNN Training (1F1B)](https://arxiv.org/abs/2004.13378), 2020
- DeepSeek-AI, [DeepSeek-V3 Technical Report (DualPipe)](https://arxiv.org/abs/2412.19437), 2024
- Korthikanti et al., [Reducing Activation Recomputation in Large Transformer Models (Sequence Parallelism)](https://arxiv.org/abs/2205.05198), 2022
- Fedus et al., [Switch Transformers (Expert Parallelism)](https://arxiv.org/abs/2101.03961), 2021
- Liu et al., [Ring Attention with Blockwise Transformers for Near-Infinite Context](https://arxiv.org/abs/2310.01889), 2023

